# Evaluate current model (v2.0) on the new DeepSeek+Kimi dataset

Benchmark: `data/labeled/deepseek_t1/labels_10ticker.final.jsonl` (30,829 T1 articles; indices zeroed, sidebar removed, persons Kimi-rescored). **Disjoint** from train/holdout.

**Two passes (per the plan + Kimi review):**
- **Phase 1 — E2E (HEADLINE):** NER predicts spans, matched to gold via char-IoU≥0.5, sentiment scored on covered entities. This is the number comparable to the production **0.6279** (e2e on the Sonnet holdout).
- **Phase 2 — Teacher-forced (DIAGNOSTIC):** gold (DeepSeek) spans fed directly, sentiment head in isolation. Phase-2 r − Phase-1 r per type tells us whether a low e2e score is a sentiment-head problem (both low) or an NER/coverage problem (TF high, e2e low).
- **Phase 3 — Gap analysis:** Spearman ρ, over-neutrality, sign-flips, bucket confusion, IoU contamination, PERSON rescored-split, calibration.

Caveats baked in: DeepSeek spans are not human-gold (NER F1 is a coverage diagnostic only; IoU distribution reported); the new labels are more conservative so MSE/MAE are distorted (Pearson/Spearman are primary).

In [ ]:
# 1. Mount Drive & check GPU (expect A100)
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Project path, checkpoint (v2.0 production), benchmark (new dataset)
import os, torch
PROJECT_PATH = "/content/drive/MyDrive/entity_sentiment_model_pipeline"
CHECKPOINT_PATH = f"{PROJECT_PATH}/trained_model/v2.0_20260517/model.pt"
BENCHMARK = f"{PROJECT_PATH}/data/labeled/deepseek_t1/labels_10ticker.final.jsonl"
OUT_DIR = f"{PROJECT_PATH}/outputs/eval_new_dataset"

assert os.path.exists(PROJECT_PATH), f"Not found: {PROJECT_PATH}"
for p in [CHECKPOINT_PATH, BENCHMARK,
          f"{PROJECT_PATH}/scripts/evaluation/evaluate_e2e_pipeline.py",
          f"{PROJECT_PATH}/scripts/evaluation/evaluate_holdout_stage3.py",
          f"{PROJECT_PATH}/scripts/evaluation/analyze_eval_gaps.py"]:
    assert os.path.exists(p), f"Missing: {p}"

# Guard against a stale checkpoint
_ck = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
_corr = _ck.get("val_metrics", {}).get("sentiment_corr", 0)
print(f"Project    : {PROJECT_PATH}")
print(f"Checkpoint : {CHECKPOINT_PATH}  (val Pearson r = {_corr:.4f})")
print(f"Benchmark  : {BENCHMARK}")
assert _corr > 0.45, f"Checkpoint looks stale (corr={_corr:.4f})"
del _ck
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 3. Install deps
!pip install -q transformers torch torchvision torchaudio
!pip install -q pytorch-crf

In [ ]:
# 4. PHASE 1 — End-to-end eval (HEADLINE; ~2-3 h on A100 for 30,829 articles)
#    --save-predictions writes per-article preds (used by Phase 3). Kept on fast
#    local disk via --local-output-dir, then synced to Drive OUT_DIR.
#    If CUDA OOM, drop --inference-batch-size to 8 or 4.
!cd {PROJECT_PATH} && python scripts/evaluation/evaluate_e2e_pipeline.py \
    --checkpoint {CHECKPOINT_PATH} \
    --benchmark {BENCHMARK} \
    --output-dir {OUT_DIR} \
    --local-output-dir /content/eval_new \
    --ner-mode single-pass \
    --inference-batch-size 16 \
    --iou-threshold 0.5 \
    --max-length 2048 \
    --log-every 200 \
    --save-predictions

In [ ]:
# 5. PHASE 2 — Teacher-forced eval (DIAGNOSTIC; gold DeepSeek spans, sentiment head only)
!cd {PROJECT_PATH} && python scripts/evaluation/evaluate_holdout_stage3.py \
    --checkpoint {CHECKPOINT_PATH} \
    --holdout {BENCHMARK} \
    --batch-size 16 \
    --device auto

In [ ]:
# 6. PHASE 3 — Gap analysis (Spearman, over-neutrality, sign-flips, bucket confusion,
#    IoU contamination, PERSON rescored-split, calibration). Reads the Phase-1 preds.
import glob
preds = sorted(glob.glob(f"/content/eval_new/e2e_predictions_*.jsonl")) or \
        sorted(glob.glob(f"{OUT_DIR}/e2e_predictions_*.jsonl"))
print("predictions:", preds[-1] if preds else "NONE")
!cd {PROJECT_PATH} && python scripts/evaluation/analyze_eval_gaps.py \
    --predictions "{preds[-1]}" \
    --final {BENCHMARK} \
    --person-scores {PROJECT_PATH}/data/labeled/deepseek_t1/person_scores.jsonl \
    --output-dir {OUT_DIR}

In [ ]:
# 7. Show results
import glob, json
m = sorted(glob.glob(f"{OUT_DIR}/e2e_metrics_*.json"))[-1]
print("=== E2E metrics:", m)
print(open(f"{OUT_DIR}/gap_analysis.md").read())

In [ ]:
# 8. (Run when done) Terminate runtime to stop billing
from google.colab import runtime
runtime.unassign()